<a href="https://colab.research.google.com/github/marius-ne/CIE_ProjectB_Group13/blob/junchao/match%20beamid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIE 2025/26 RWTH, PROJECT B, GROUP 13
Junchao Yu, Marius Neuhalfen

ToDo:
- Find defective nodes by comparing perfect structure and imperfect structures for all scenarios
- Group defective nodes into regions (arc sections or track sections)
- Predict whether structure is perfect or imperfect
- Predict where the imperfection lies

There are 25.XXX for deformation and 24.XXX for stress

#change I-beam name

In [15]:
import pandas as pd
import re
import plotly.express as px

def process_ibeam_data():
    input_file = 'I-beam.txt'
    output_file = 'I-beam name.txt'

    # 1. 读取并修复格式（将 "body 38" 转换为 "body_38"）
    try:
        with open(input_file, 'r') as f:
            content = f.read()

        # 正则替换：匹配行首或换行后的 "body " + 数字，替换为 "body_"
        cleaned_content = re.sub(r'(^|\n)body (\d+)', r'\1body_\2', content)

        with open(output_file, 'w') as f:
            f.write(cleaned_content)
        print(f"已生成格式统一的文件: {output_file}")

    except FileNotFoundError:
        print(f"错误: 找不到文件 {input_file}")
        return

    # 2. 读取清洗后的数据
    # on_bad_lines='skip' 防止极少数极端格式错误导致崩溃，但上面的修复通常已解决问题
    df = pd.read_csv(output_file, sep='\s+', on_bad_lines='skip')

    # 确保坐标列为数值型
    cols = ['X', 'Y', 'Z']
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # 删除坐标缺失的行
    df = df.dropna(subset=['X', 'Y', 'Z'])

    # 3. 筛选目标 Body
    # 定义三个区间
    ranges = [
        (0.049803, 0.098988),
        (0.1498, 0.19899),
        (0.2498, 0.29899)
    ]

    # 创建掩码（使用 vector 运算避免索引问题）
    mask = pd.Series(False, index=df.index)
    epsilon = 1e-6 # 浮点数容差
    for r in ranges:
        mask = mask | ((df['Y'] >= r[0] - epsilon) & (df['Y'] <= r[1] + epsilon))

    filtered_df = df[mask].copy()
    unique_bodies = filtered_df['BodyName'].unique()
    print(f"筛选出 {len(unique_bodies)} 个符合条件的 Body。")

    # 4. 排序逻辑
    # "将筛选出的body name将x坐标相同，y坐标从小到大的顺序"
    # 我们计算每个 Body 的重心（平均 X 和 Y）来代表它
    body_stats = []
    for body in unique_bodies:
        subset = filtered_df[filtered_df['BodyName'] == body]
        body_stats.append({
            'BodyName': body,
            'mean_x': subset['X'].mean(),
            'mean_y': subset['Y'].mean()
        })

    stats_df = pd.DataFrame(body_stats)

    # 为了处理浮点数误差，将 X 坐标四舍五入到 3 位小数进行“分组”
    stats_df['x_group'] = stats_df['mean_x'].round(3)

    # 排序：先按 X 分组（从小到大），再按 Y 从小到大
    stats_df = stats_df.sort_values(by=['x_group', 'mean_y'])

    sorted_bodies = stats_df['BodyName'].tolist()

    # 5. 分组 (每 8 个一组)
    id_need_change = [sorted_bodies[i:i+8] for i in range(0, len(sorted_bodies), 8)]

    # 输出结果到控制台
    print(f"共分成 {len(id_need_change)} 组 (id_need_change):")
    for idx, group in enumerate(id_need_change):
        print(f"Group {idx+1}: {group}")

    # 6. 3D 交互式绘图
    if not filtered_df.empty:
        fig = px.scatter_3d(
            filtered_df,
            x='X', y='Y', z='Z',
            color='BodyName',
            hover_name='BodyName',
            title='Filtered Bodies 3D Visualization',
            category_orders={'BodyName': sorted_bodies} # 图例按照我们的排序显示
        )

        # 不压缩坐标轴 (aspectmode='data')
        fig.update_layout(
            scene=dict(
                aspectmode='data',
                xaxis_title='X',
                yaxis_title='Y',
                zaxis_title='Z'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )

        fig.show()

    return id_need_change

# 执行函数
id_need_change = process_ibeam_data()

<>:27: SyntaxWarning:

invalid escape sequence '\s'

<>:27: SyntaxWarning:

invalid escape sequence '\s'

/tmp/ipython-input-1401125072.py:27: SyntaxWarning:

invalid escape sequence '\s'



已生成格式统一的文件: I-beam name.txt
筛选出 72 个符合条件的 Body。
共分成 9 组 (id_need_change):
Group 1: ['Body_416', 'Body_362', 'Body_308', 'Body_254', 'Body_200', 'Body_146', 'Body_92', 'body_38']
Group 2: ['Body_425', 'Body_371', 'Body_317', 'Body_263', 'Body_209', 'Body_155', 'Body_101', 'body_47']
Group 3: ['Body_434', 'Body_380', 'Body_326', 'Body_272', 'Body_218', 'Body_164', 'Body_110', 'body_56']
Group 4: ['Body_419', 'Body_365', 'Body_311', 'Body_257', 'Body_203', 'Body_149', 'Body_95', 'body_41']
Group 5: ['Body_428', 'Body_374', 'Body_320', 'Body_266', 'Body_212', 'Body_158', 'Body_104', 'body_50']
Group 6: ['Body_437', 'Body_383', 'Body_329', 'Body_275', 'Body_221', 'Body_167', 'Body_113', 'body_59']
Group 7: ['body_44', 'body_53', 'body_62', 'Body_422', 'Body_368', 'Body_314', 'Body_260', 'Body_206']
Group 8: ['Body_152', 'Body_98', 'Body_431', 'Body_377', 'Body_323', 'Body_269', 'Body_215', 'Body_161']
Group 9: ['Body_107', 'Body_440', 'Body_386', 'Body_332', 'Body_278', 'Body_224', 'Body_17

In [22]:
import pandas as pd
import re
import plotly.express as px
import numpy as np

def process_and_visualize_beam():
    # --- 1. 读取并修复原始文件格式 ---
    input_file = 'I-beam.txt'

    try:
        with open(input_file, 'r') as f:
            content = f.read()
        # 修复 "body 38" 为 "body_38" 以便统一处理
        cleaned_content = re.sub(r'(^|\n)body (\d+)', r'\1body_\2', content)

        # 使用 StringIO 模拟文件读取
        from io import StringIO
        df = pd.read_csv(StringIO(cleaned_content), sep='\s+', on_bad_lines='skip')

        # 确保坐标列为数值
        for col in ['X', 'Y', 'Z']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna(subset=['X', 'Y', 'Z'])

    except FileNotFoundError:
        print(f"错误: 找不到文件 {input_file}")
        return

    # --- 2. 筛选与排序 (生成 id_need_change) ---
    # 定义区间
    ranges = [(0.0498, 0.099), (0.1498, 0.199), (0.2498, 0.299)]

    # 筛选
    mask = pd.Series(False, index=df.index)
    for r in ranges:
        mask = mask | ((df['Y'] >= r[0]) & (df['Y'] <= r[1]))

    filtered_df = df[mask].copy()

    # 排序：先按 X (四舍五入分组) 从小到大，再按 Y 从小到大
    # 计算重心用于排序
    body_stats = filtered_df.groupby('BodyName')[['X', 'Y']].mean().reset_index()
    body_stats['x_group'] = body_stats['X'].round(3)
    sorted_bodies = body_stats.sort_values(by=['x_group', 'Y'])['BodyName'].tolist()

    # 分组：每 8 个一组
    id_need_change = [sorted_bodies[i:i+8] for i in range(0, len(sorted_bodies), 8)]
    print(f"筛选并分组完毕，共 {len(id_need_change)} 组。")

    # --- 3. 数值运算与重命名逻辑 ---
    # 定义每组 8 个元素对应的运算规则
    offsets = [-27, 54, 54, 54, 54, 54, 54, 54]

    rename_map = {}

    # A. 处理在 id_need_change 中的 Body
    for group in id_need_change:
        for idx, body_name in enumerate(group):
            if idx < len(offsets):
                # 提取原始数字 (例如 Body_101 -> 101)
                num_match = re.search(r'(\d+)', body_name)
                if num_match:
                    original_num = int(num_match.group(1))
                    new_num = original_num + offsets[idx] # 执行运算
                    rename_map[body_name] = f"Beam_{new_num}"

    # B. 处理剩余所有 Body (包含未被筛选的)
    # 逻辑：如果在 map 中则用新算出的名字，否则保留原数字但改前缀为 Beam
    def get_new_beam_name(old_name):
        if old_name in rename_map:
            return rename_map[old_name]
        else:
            # 提取数字，保留原样，仅改前缀
            num_match = re.search(r'(\d+)', old_name)
            if num_match:
                return f"Beam_{num_match.group(1)}"
            return f"Beam_{old_name}" # 兜底

    # 应用重命名
    df['BodyName'] = df['BodyName'].apply(get_new_beam_name)

    # --- 4. 修改列名并保存 ---
    # 将 'BodyName' 列重命名为 'Beam'
    df = df.rename(columns={'BodyName': 'Beam'})

    # 保存文件
    output_filename = 'I beam after change.txt'
    # sep='\t' 使用制表符分隔，或用空格 sep=' '，保持与原文件类似格式
    df.to_csv(output_filename, sep='\t', index=False)
    print(f"文件已保存: {output_filename} (列名已修改为 Beam)")

    # --- 5. 3D 交互式可视化 ---
    print("正在生成 3D 图像...")

    # 注意：这里使用新的列名 'Beam'
    fig = px.scatter_3d(
        df,
        x='X', y='Y', z='Z',
        color='Beam',          # 颜色依据新的 Beam 列
        hover_name='Beam',     # 悬停显示新的 Beam 列
        title='I Beam After Change (Interactive)',
        opacity=1.0
    )

    # 设置不压缩坐标轴
    fig.update_layout(
        scene=dict(
            aspectmode='data', # 保持真实比例
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z'
        ),
        margin=dict(l=0, r=0, b=0, t=30)
    )

    fig.show()

# 执行
if __name__ == "__main__":
    process_and_visualize_beam()

<>:18: SyntaxWarning:

invalid escape sequence '\s'

<>:18: SyntaxWarning:

invalid escape sequence '\s'

/tmp/ipython-input-1543222842.py:18: SyntaxWarning:

invalid escape sequence '\s'



筛选并分组完毕，共 9 组。
文件已保存: I beam after change.txt (列名已修改为 Beam)
正在生成 3D 图像...


#change x-beam name

In [23]:
import pandas as pd

def format_xbeam_file():
    input_file = 'X-beam-final_named_structure.txt'
    output_file = 'x beam after change.txt'

    try:
        # 读取 X-beam 文件 (假设是用制表符分隔，如果是空格则改 sep='\s+')
        df = pd.read_csv(input_file, sep='\t')

        # 1. 修改列名: BodyName -> Beam
        if 'BodyName' in df.columns:
            df = df.rename(columns={'BodyName': 'Beam'})

        # 2. 统一名字格式: 将 Body_XXX 替换为 Beam_XXX
        # 确保数据是字符串类型再进行替换
        if 'Beam' in df.columns:
            df['Beam'] = df['Beam'].astype(str).str.replace('Body_', 'Beam_', regex=False)

            # 如果还有其他格式（比如 'body 38' 这种带空格的），也可以一并处理：
            # df['Beam'] = df['Beam'].str.replace('body ', 'Beam_', regex=False, case=False)

        # 3. 保存文件
        # index=False 不保存行索引
        # sep='\t' 保持与 I-beam 文件一致的制表符分隔
        df.to_csv(output_file, sep='\t', index=False)

        print(f"处理完成！文件已保存为: {output_file}")
        print("前5行预览：")
        print(df.head())

    except FileNotFoundError:
        print(f"错误：找不到文件 {input_file}")
    except Exception as e:
        print(f"发生错误: {e}")

# 运行函数
if __name__ == '__main__':
    format_xbeam_file()

处理完成！文件已保存为: x beam after change.txt
前5行预览：
       Beam   Node         X        Y         Z
0  Beam_101  16064  0.001888  0.19680  0.011712
1  Beam_101  16065  0.003794  0.19630  0.012169
2  Beam_101  16066  0.005686  0.19580  0.012614
3  Beam_101  16067  0.007595  0.19530  0.013055
4  Beam_101  16068  0.005705  0.19481  0.013482


#merge X and I beams


In [36]:
import pandas as pd
import plotly.express as px

def sort_and_visualize_final():
    input_file = 'merge I and X.txt'
    output_file = 'merge I and X sorted.txt'

    try:
        # 1. 读取数据
        df = pd.read_csv(input_file, sep='\t')

        # 2. 计算每个 Beam 的重心 (用于排序)
        beam_stats = df.groupby('Beam')[['X', 'Y']].mean().reset_index()

        # 3. 定义排序逻辑：从左上(Top-Left) 到 右下(Bottom-Right)
        # "按行" (Row by row) 通常指 Y 轴的分层。
        # "Top" (上) 在数学坐标系中通常指 Y 值较大；"Left" (左) 指 X 值较小。

        # 为了处理浮点数误差，将 Y 进行分箱/取整，定义"行"
        # 例如 0.28 和 0.29 应该属于同一行。我们保留1位小数进行分组即可分辨层级。
        beam_stats['y_row'] = beam_stats['Y'].round(1)

        # 排序：
        # Primary: y_row 降序 (从上到下)
        # Secondary: X 升序 (从左到右)
        sorted_beams_df = beam_stats.sort_values(
            by=['y_row', 'X'],
            ascending=[False, True]
        )

        sorted_beam_names = sorted_beams_df['Beam'].tolist()

        print(f"排序完成。共 {len(sorted_beam_names)} 个 Beam。")
        print("前 5 个 Beam (左上角):", sorted_beam_names[:5])
        print("后 5 个 Beam (右下角):", sorted_beam_names[-5:])

        # 4. 根据新的 Beam 顺序重排原始数据
        # 将 Beam 列转换为 Categorical 类型，并指定顺序
        df['Beam'] = pd.Categorical(df['Beam'], categories=sorted_beam_names, ordered=True)

        # 执行排序
        df_sorted = df.sort_values(by=['Beam'])

        # 5. 保存文件
        df_sorted.to_csv(output_file, sep='\t', index=False)
        print(f"文件已保存至: {output_file}")

        # 6. 3D 可视化
        print("正在生成 3D 图像...")

        fig = px.scatter_3d(
            df_sorted,
            x='X', y='Y', z='Z',
            color='Beam',
            hover_name='Beam', # 光标悬停显示 Beam 名称
            hover_data={'X':':.4f', 'Y':':.4f', 'Z':':.4f', 'Beam':False},
            title='Merge I and X (Sorted: Top-Left to Bottom-Right)',
            opacity=1.0 # 不透明
        )

        # 设置点的大小和样式
        fig.update_traces(marker=dict(size=3))

        # 强制不压缩坐标轴
        fig.update_layout(
            scene=dict(
                aspectmode='data', # 保持真实比例
                xaxis_title='X (Left -> Right)',
                yaxis_title='Y (Bottom -> Top)', # 注意 Plotly Y轴默认方向
                zaxis_title='Z'
            ),
            margin=dict(l=0, r=0, b=0, t=30),
            # 强制图例顺序与我们排序的顺序一致
            legend={'traceorder': 'normal'}
        )

        fig.show()

    except FileNotFoundError:
        print(f"错误: 找不到文件 {input_file}")
    except Exception as e:
        print(f"发生错误: {e}")

if __name__ == "__main__":
    sort_and_visualize_final()

排序完成。共 135 个 Beam。
前 5 个 Beam (左上角): ['Beam_110', 'Beam_218', 'Beam_164', 'Beam_272', 'Beam_434']
后 5 个 Beam (右下角): ['Beam_179', 'Beam_341', 'Beam_233', 'Beam_395', 'Beam_287']
文件已保存至: merge I and X sorted.txt
正在生成 3D 图像...


#lable track and arch

In [37]:
import pandas as pd

# 1. 读取文件
# 假设文件是以制表符分隔的 (根据之前的文件格式)
file_path = 'merge I and X sorted.txt'
df = pd.read_csv(file_path, sep='\t')

# 2. 定义分类函数
def assign_track(x):
    # 根据 X 坐标所在的区间进行分类
    # 第一类: (0, 0.019) -> 1 (小于 0.02)
    if x < 0.02:
        return 1
    # 第二类: (0.02, 0.039) -> 2 (0.02 到 0.04 之间)
    elif x < 0.04:
        return 2
    # 第三类: (0.04, 0.059) -> 3 (大于 0.04)
    else:
        return 3

# 3. 应用函数创建新列 'track'
df['track'] = df['X'].apply(assign_track)

# 4. 打印预览
print("添加 track 列后的数据预览：")
print(df.head())

# 5. 保存结果到新文件
output_file = 'merge I and X track.txt'
df.to_csv(output_file, sep='\t', index=False)
print(f"\n已保存文件为: {output_file}")

添加 track 列后的数据预览：
       Beam   Node         X        Y         Z  track
0  Beam_110  16396  0.017097  0.29382  0.015596      1
1  Beam_110  16367  0.004759  0.29356  0.014523      1
2  Beam_110  16361  0.014241  0.29356  0.014523      1
3  Beam_110  16387  0.014241  0.29356  0.015807      1
4  Beam_110  16330  0.005686  0.29580  0.012614      1

已保存文件为: merge I and X track.txt


In [38]:
import pandas as pd

# 1. 读取文件
# 假设文件是以制表符分隔的
file_path = 'merge I and X track.txt'
df = pd.read_csv(file_path, sep='\t')

# 2. 定义分类函数
def assign_arch(y):
    # 第一类: (0.0008, 0.099)
    # 考虑到浮点数精度和数据间隙，我们使用 < 0.1 作为分界
    if y < 0.1:
        return 1
    # 第二类: (0.1008, 0.199)
    elif y < 0.2:
        return 2
    # 第三类: (0.2008, 0.299)
    else:
        return 3

# 3. 应用函数创建新列 'arch'
df['arch'] = df['Y'].apply(assign_arch)

# 4. 打印预览
print("添加 arch 列后的数据预览：")
print(df.head())
print("\n各类别的数量统计：")
print(df['arch'].value_counts())

# 5. 保存结果到新文件
output_file = 'merge I and X track arch.txt'
df.to_csv(output_file, sep='\t', index=False)
print(f"\n已保存文件为: {output_file}")

添加 arch 列后的数据预览：
       Beam   Node         X        Y         Z  track  arch
0  Beam_110  16396  0.017097  0.29382  0.015596      1     3
1  Beam_110  16367  0.004759  0.29356  0.014523      1     3
2  Beam_110  16361  0.014241  0.29356  0.014523      1     3
3  Beam_110  16387  0.014241  0.29356  0.015807      1     3
4  Beam_110  16330  0.005686  0.29580  0.012614      1     3

各类别的数量统计：
arch
3    2671
2    2671
1    2671
Name: count, dtype: int64

已保存文件为: merge I and X track arch.txt


In [39]:
import pandas as pd

# 1. 读取文件
# 假设文件是以制表符分隔的
file_path = 'merge I and X track arch.txt'
df = pd.read_csv(file_path, sep='\t')

# 2. 确定 Beam 的排序依据
# 为了让同一个 Beam 的点排在一起，并且 Beam 之间按位置排序，
# 我们计算每个 Beam 的全局最小 Y 值 (beam_min_y)
beam_min_y = df.groupby('Beam')['Y'].min().rename('beam_min_y')

# 将这个排序键合并回主数据
df = df.merge(beam_min_y, on='Beam')

# 3. 执行排序
# 优先级顺序：
# Level 1: track (从低到高，将数据分组)
# Level 2: arch (从低到高，将数据分组)
# Level 3: beam_min_y (从小到大，确保每组内的 Beam 按物理位置排列)
# Level 4: Beam (字符串排序，仅作为 Y 值完全相同时的次要依据)
# Level 5: Y (Beam 内部的点按 Y 排列)
# Level 6: X (Beam 内部的点按 X 排列)
sorted_df = df.sort_values(
    by=['track', 'arch', 'beam_min_y', 'Beam', 'Y', 'X'],
    ascending=[True, True, True, True, True, True]
)

# 4. 删除辅助列 (可选)
sorted_df = sorted_df.drop(columns=['beam_min_y'])

# 5. 保存结果
output_file = 'merge I and X track arch sorted.txt'
sorted_df.to_csv(output_file, sep='\t', index=False)

print(f"排序完成！文件已保存为: {output_file}")
print("前 10 行数据预览：")
print(sorted_df.head(10))

排序完成！文件已保存为: merge I and X track arch sorted.txt
前 10 行数据预览：
         Beam   Node         X         Y         Z  track  arch
6671   Beam_1  24746  0.000000  0.000810  0.010193      1     1
6672   Beam_1  24749  0.004750  0.000810  0.010193      1     1
6675   Beam_1  24748  0.009500  0.000810  0.010193      1     1
6673   Beam_1  24747  0.014250  0.000810  0.010193      1     1
6674   Beam_1  24745  0.019000  0.000810  0.010193      1     1
6686  Beam_65  15043  0.001574  0.001723  0.010308      1     1
6705  Beam_65  15069  0.001574  0.001723  0.011699      1     1
6704  Beam_65  15046  0.017426  0.001723  0.010308      1     1
6708  Beam_65  15059  0.017426  0.001723  0.011699      1     1
6682  Beam_65  15042  0.003161  0.002139  0.010713      1     1


#rename

In [41]:
import pandas as pd

# 1. 读取文件
file_path = 'merge I and X track arch sorted.txt'
df = pd.read_csv(file_path, sep='\t')

# 2. 定义标签生成逻辑
def assign_h_labels(df):
    # 使用 sort=False 保持 groupby 后的顺序与原文件一致（原文件已排序）
    groups = df.groupby(['track', 'arch'], sort=False)

    annotated_chunks = []

    for (track, arch), group in groups:
        # 获取该组内所有唯一的 Beam，保持出现顺序（即从上至下）
        unique_beams = group['Beam'].unique()

        # 创建 Beam 到 H 标签的映射
        mapping = {}
        for i, beam in enumerate(unique_beams):
            # 逻辑：第1个标为2，第2个标为1，其余标为序号+1
            if i == 0:
                label = 2
            elif i == 1:
                label = 1
            else:
                label = i + 1
            mapping[beam] = label

        # 将标签应用到该组数据
        group_with_h = group.copy()
        group_with_h['H'] = group['Beam'].map(mapping)
        annotated_chunks.append(group_with_h)

    # 将所有分组合并回一个 DataFrame
    return pd.concat(annotated_chunks)

# 3. 执行处理
df_final = assign_h_labels(df)

# 4. 打印预览
print("添加 H 列后的数据预览：")
print(df_final[['Beam', 'track', 'arch', 'H']].head(10))

# 检查某个组的标签分配情况（例如 track=1, arch=2）
print("\n检查 Track 1, Arch 2 组的 Beam 标签分配：")
check_group = df_final[(df_final['track']==1) & (df_final['arch']==2)]
print(check_group[['Beam', 'H']].drop_duplicates())

# 5. 保存结果
output_file = 'renamed beam.txt'
df_final.to_csv(output_file, sep='\t', index=False)
print(f"\n文件已保存为: {output_file}")

添加 H 列后的数据预览：
      Beam  track  arch  H
0   Beam_1      1     1  2
1   Beam_1      1     1  2
2   Beam_1      1     1  2
3   Beam_1      1     1  2
4   Beam_1      1     1  2
5  Beam_65      1     1  1
6  Beam_65      1     1  1
7  Beam_65      1     1  1
8  Beam_65      1     1  1
9  Beam_65      1     1  1

检查 Track 1, Arch 2 组的 Beam 标签分配：
          Beam   H
890     Beam_5   2
894    Beam_74   1
971   Beam_128   3
1030  Beam_182   4
1091  Beam_236   5
1152  Beam_290   6
1213  Beam_344   7
1274  Beam_398   8
1335  Beam_425   9
1396  Beam_371  10
1457  Beam_317  11
1518  Beam_263  12
1579  Beam_209  13
1640  Beam_155  14
1701  Beam_101  15

文件已保存为: renamed beam.txt


In [42]:
import pandas as pd

# 1. 读取文件
file_path = 'renamed beam.txt'
df = pd.read_csv(file_path, sep='\t')

# 2. 提取 Beam 中的原始数字
# 假设格式为 "Beam_123"，提取 "123"
df['original_number'] = df['Beam'].str.extract(r'(\d+)')

# 3. 生成新的后缀字符串
# 格式: track_arch_H (例如 1_1_2)
new_suffix = (
    df['track'].astype(str) + '_' +
    df['arch'].astype(str) + '_' +
    df['H'].astype(str)
)

# 4. 更新 Beam 列
# 将 "Beam_123" 替换为 "Beam_1_1_2"
# 我们保留 "Beam_" 前缀 (假设所有行前缀一致)
prefix = df['Beam'].iloc[0].split('_')[0] # 获取前缀 (如 "Beam")
df['Beam'] = prefix + '_' + new_suffix

# 5. 将原始数字移动到 H 列
df['H'] = df['original_number']

# 6. 将 H 列重命名为 original
df.rename(columns={'H': 'original'}, inplace=True)

# 7. 清理临时列
df.drop(columns=['original_number'], inplace=True)

# 8. 打印预览
print("处理后的数据预览：")
print(df[['Beam', 'track', 'arch', 'original']].head())

# 9. 保存结果
output_file = 'new beam.txt'
df.to_csv(output_file, sep='\t', index=False)
print(f"\n已保存文件为: {output_file}")

处理后的数据预览：
         Beam  track  arch original
0  Beam_1_1_2      1     1        1
1  Beam_1_1_2      1     1        1
2  Beam_1_1_2      1     1        1
3  Beam_1_1_2      1     1        1
4  Beam_1_1_2      1     1        1

已保存文件为: new beam.txt


In [44]:
import pandas as pd
import plotly.express as px

# 1. 读取数据
file_path = 'new beam.txt'
# 假设文件是以制表符分隔的
df = pd.read_csv(file_path, sep='\t')

# 2. 设置点的大小 (您可以修改此数值)
point_size = 2

# 3. 创建 3D 散点图
# 注意：我们将 Beam 转换为字符串，以确保每个 Beam 有独立的颜色
df['Beam_Str'] = df['Beam'].astype(str)

fig = px.scatter_3d(
    df,
    x='X',
    y='Y',
    z='Z',
    color='Beam_Str',  # 按 Beam 分色
    hover_name='Node', # 鼠标悬停标题显示 Node Number
    hover_data={       # 自定义悬停显示的数据
        'Beam': True,
        'Beam_Str': False, # 隐藏辅助列
        'X': True,
        'Y': True,
        'Z': True,
        'original': True # 显示原始 Beam ID
    },
    title='New Beam 3D Visualization',
    labels={'Beam_Str': 'Beam ID'}
)

# 4. 关键设置：真实比例与点大小
fig.update_traces(marker=dict(size=point_size)) # 应用点大小

fig.update_layout(
    scene=dict(
        aspectmode='data', # 强制坐标轴比例一致 (不压缩)
        xaxis_title='X Axis',
        yaxis_title='Y Axis',
        zaxis_title='Z Axis'
    ),
    margin=dict(l=0, r=0, b=0, t=40) # 优化边距
)

# 5. 显示图像
fig.show()